# W2 — Training (Kaggle GPU)

**Trước khi chạy:** Settings → *Internet: ON*, *Accelerator: **GPU T4 x1** (hoặc P100)*.
**Add Data →** dataset `lits-processed` (từ W1: `images_u8.npy` + `manifest.csv` + `splits/`).

Luồng: verify → train baseline **ResNet-50** (fold 0) → eval patient-level (AUROC + CI + ROC/PR/calibration).
Đổi `--arch convnextv2_nano` để chạy model chính; các backbone khác xem `configs/train/base.yaml`.

In [ ]:
# 1) Thư viện (Kaggle có sẵn torch/pandas/sklearn/matplotlib)
!pip install -q -U timm && pip install -q mlflow albumentations

In [ ]:
# 2) Nạp code + set DATA_ROOT (tự dò folder chứa images_u8.npy — bền với mount path)
import os, glob
if not os.path.exists('HCC-TACE-Assist'):
    !git clone -q https://github.com/hdtruong802/HCC-TACE-Assist.git
%cd HCC-TACE-Assist
import sys; sys.path.insert(0, '.')
_hits = glob.glob('/kaggle/input/**/images_u8.npy', recursive=True)
DATA_ROOT = os.path.dirname(_hits[0]) if _hits else '/kaggle/input/lits-processed'
os.environ['DATA_ROOT'] = DATA_ROOT
print('DATA_ROOT =', DATA_ROOT, '| tồn tại:', os.path.exists(DATA_ROOT))
if not _hits:
    print('⚠ Chưa thấy images_u8.npy → Add Data lits-processed rồi RESTART SESSION (Run → Restart), chạy lại.')
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3) VERIFY: manifest + split + mảng ảnh khớp nhau
import pandas as pd, numpy as np
from src.data.dataset import resolve, load_split
R = os.environ['DATA_ROOT']
df = pd.read_csv(resolve(R, 'manifest.csv'))
arr = np.load(resolve(R, 'images_u8.npy'), mmap_mode='r')
sp = load_split(resolve(R, 'splits/lits_v1.json'))   # tự tìm ở dataset hoặc repo
print('slices:', len(df), '| images:', arr.shape, '| patients:', df.patient_id.nunique())
print('folds:', len(sp['folds']), '| test bn:', len(sp['test']), '| pos_ratio:', round((df.label==1).mean(),3))
assert arr.shape[0] == len(df), 'images_u8.npy KHONG khop manifest!'

In [ ]:
# 4) TRAIN baseline ResNet-50 (fold 0) — dùng DATA_ROOT đã dò (env)
!python -m src.training.train --config configs/train/base.yaml --arch resnet50 --fold 0

In [ ]:
# 5) EVAL trên validation (patient-level) + curves
!python -m src.evaluation.evaluate --config configs/train/base.yaml \
    --ckpt /kaggle/working/outputs/resnet50_fold0/best.ckpt --split val --fold 0
from IPython.display import Image, display
for f in ['roc','pr','confusion','calibration']:
    p = f'/kaggle/working/outputs/resnet50_fold0/eval_val/{f}.png'
    if os.path.exists(p): display(Image(p))

In [ ]:
# 6) GRAD-CAM sanity — model bam dung o ton thuong hay "shortcut"? (can lits-processed)
import os, glob
ARCH = 'convnextv2_nano'
def find_ckpt(arch, fold=0):
    c = [f'/kaggle/working/outputs/{arch}_fold{fold}/best.ckpt']
    c += sorted(glob.glob(f'/kaggle/input/**/{arch}_fold{fold}*.ckpt', recursive=True))
    c += sorted(glob.glob(f'/kaggle/input/**/{arch}*fold{fold}*.ckpt', recursive=True))
    c += sorted(glob.glob('/kaggle/input/**/*.ckpt', recursive=True))
    return next((x for x in c if os.path.exists(x)), None)
CKPT = find_ckpt(ARCH); print('CKPT =', CKPT)
assert CKPT, 'Khong tim thay .ckpt — Add Data dataset chua checkpoint, hoac train cell 4 truoc.'
OUT = f'/kaggle/working/eval_out/{ARCH}'
!cd /kaggle/working/HCC-TACE-Assist && python -m src.interpretability.gradcam --config configs/train/base.yaml --ckpt {CKPT} --split val --fold 0 --n-per 6 --out {OUT}/gradcam
from IPython.display import Image, display
g = f'{OUT}/gradcam/gradcam_val.png'
if os.path.exists(g): display(Image(g))

In [ ]:
# 7) EXTERNAL 3D-IRCADb-01 — VERIFY cấu trúc (nhớ Add Data dataset ircad vào notebook)
import os, glob
try:
    import pydicom
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q','pydicom'])
    import pydicom
hits = glob.glob('/kaggle/input/**/PATIENT_DICOM', recursive=True)
print('So benh nhan (PATIENT_DICOM):', len(hits))
if hits:
    p0 = os.path.dirname(sorted(hits)[0]); print('Vi du:', p0)
    md = os.path.join(p0,'MASKS_DICOM')
    print('MASKS_DICOM:', sorted(os.listdir(md)) if os.path.isdir(md) else 'KHONG co')
    f0 = sorted(f for f in glob.glob(os.path.join(p0,'PATIENT_DICOM','*')) if os.path.isfile(f))[0]
    d = pydicom.dcmread(f0, force=True)
    sl=float(getattr(d,'RescaleSlope',1) or 1); ic=float(getattr(d,'RescaleIntercept',0) or 0)
    a=d.pixel_array
    print('1 slice shape', a.shape, '| HU ~', round(a.min()*sl+ic), '..', round(a.max()*sl+ic))
else:
    print('CHUA thay -> Add Data dataset 3dircadb1 roi Restart & Run.')

In [ ]:
# 8) BUILD IRCADb cache + manifest (CPU, vai phut) — CUNG preprocess/tau voi LiTS
!cd /kaggle/working/HCC-TACE-Assist && python scripts/build_ircad.py --data-root /kaggle/input --out-dir /kaggle/working

In [ ]:
# 9) EVAL external (cham 1 LAN) — threshold KHOA tu val. Ghi ket qua vao /kaggle/working.
import os, glob
ARCH = 'convnextv2_nano'
def find_ckpt(arch, fold=0):
    c = [f'/kaggle/working/outputs/{arch}_fold{fold}/best.ckpt']
    c += sorted(glob.glob(f'/kaggle/input/**/{arch}_fold{fold}*.ckpt', recursive=True))
    c += sorted(glob.glob(f'/kaggle/input/**/{arch}*fold{fold}*.ckpt', recursive=True))
    c += sorted(glob.glob('/kaggle/input/**/*.ckpt', recursive=True))
    return next((x for x in c if os.path.exists(x)), None)
CKPT = find_ckpt(ARCH); print('CKPT =', CKPT)
assert CKPT, 'Khong tim thay .ckpt — Add Data dataset chua checkpoint, hoac train cell 4 truoc.'
OUT = f'/kaggle/working/eval_out/{ARCH}/eval_ircad'
!cd /kaggle/working/HCC-TACE-Assist && python -m src.evaluation.eval_external --config configs/train/base.yaml --ircad-config configs/data/ircad.yaml --ckpt {CKPT} --data-root /kaggle/working --out {OUT}
from IPython.display import Image, display
for f in ['roc_slice','pr_slice','roc','pr','confusion','calibration']:
    q=f'{OUT}/{f}.png'
    if os.path.exists(q): display(Image(q))
# Tai {OUT}/metrics.json + png ve image_eval/ircad_convnextv2_nano/ de dua vao bao cao.

## Bước tiếp
- **Model chính:** đổi cell 4 → `--arch convnextv2_nano` (hoặc `convnextv2_tiny`). So sánh AUROC val.
- **So sánh nhanh (Pha 1):** chạy mỗi arch 1 lần (fold 0) → xếp hạng theo `val_auroc`.
- **Pha 2 (nghiêm ngặt):** 1–2 model top × 5 fold (`--fold 0..4`) × nhiều seed → gộp CI. *(dùng FPT H100 nếu Kaggle hết quota)*.
- **Internal test (1 lần):** `--split test` với ngưỡng đã khóa trên val.
- Lưu `outputs/` (ckpt + metrics + curves); tải `val_metrics.json` về commit để so sánh.